# Week 8 Hands-on — Attention Visualization

In this notebook, we build an intuition for **attention** and **self-attention** using small examples and visualizations.

The goal is not to train a Transformer from scratch. The goal is to understand what an attention matrix represents and how tokens can attend to other tokens.

## Learning Objectives

By the end of this notebook, you should be able to:
- explain what an attention matrix is
- interpret attention weights
- visualize attention using heatmaps
- understand how self-attention differs from RNN-style sequential processing
- explain why attention is useful for Information Extraction tasks such as NER


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Example Sentence

We use a short sentence that contains an organization and a location.

```text
Amazon opened a new office in Paris.
```

For an Information Extraction system, the important entities are:

- `Amazon` → ORG
- `Paris` → GPE / LOC


In [ ]:
tokens = ["Amazon", "opened", "a", "new", "office", "in", "Paris", "."]
tokens

## 2. What is an Attention Matrix?

An attention matrix tells us how much each token attends to every other token.

Rows represent the **query token** — the token that is looking for information.

Columns represent the **tokens being attended to**.

Each row usually sums to 1.

For example, if the row for `Paris` gives high weight to `office`, this means that the model uses the word `office` as context when representing `Paris`.

In [ ]:
attention = np.array([
    [0.40, 0.15, 0.05, 0.05, 0.15, 0.05, 0.10, 0.05],  # Amazon
    [0.25, 0.30, 0.05, 0.05, 0.15, 0.05, 0.10, 0.05],  # opened
    [0.10, 0.10, 0.35, 0.15, 0.10, 0.05, 0.10, 0.05],  # a
    [0.10, 0.10, 0.10, 0.30, 0.20, 0.05, 0.10, 0.05],  # new
    [0.20, 0.20, 0.05, 0.10, 0.25, 0.05, 0.10, 0.05],  # office
    [0.10, 0.10, 0.05, 0.05, 0.20, 0.20, 0.25, 0.05],  # in
    [0.25, 0.10, 0.03, 0.05, 0.30, 0.10, 0.12, 0.05],  # Paris
    [0.10, 0.10, 0.05, 0.05, 0.15, 0.05, 0.10, 0.40],  # .
])

attention.sum(axis=1)

## 3. Visualize the Attention Matrix

A heatmap makes attention easier to interpret.

Darker cells mean higher attention weights.

In [ ]:
def plot_attention_matrix(attention, tokens, title="Attention Matrix"):
    plt.figure(figsize=(8, 6))
    plt.imshow(attention)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel("Attended token")
    plt.ylabel("Query token")
    plt.title(title)
    plt.colorbar(label="Attention weight")
    plt.tight_layout()
    plt.show()


plot_attention_matrix(attention, tokens)

## 4. Which Tokens Does `Paris` Attend To?

Now we inspect the attention row for the token `Paris`.

This helps us understand which context words influence the representation of `Paris`.

In [ ]:
query_token = "Paris"
query_index = tokens.index(query_token)

paris_attention = pd.DataFrame({
    "attended_token": tokens,
    "attention_weight": attention[query_index]
}).sort_values("attention_weight", ascending=False)

paris_attention

### Discussion

Look at the highest attention weights for `Paris`.

Questions:

1. Which words receive the highest attention?
2. Do these words help identify `Paris` as a location?
3. Would this be possible with a fixed regex rule?


## 5. RNN-style Processing vs Self-Attention

An RNN processes tokens sequentially:

```text
Amazon → opened → a → new → office → in → Paris
```

Information from `Amazon` must pass through many hidden states before reaching `Paris`.

Self-attention is different:

```text
Every token can directly attend to every other token.
```

This makes it easier to capture long-range dependencies.

In [ ]:
distance_rows = []

for i, token_i in enumerate(tokens):
    for j, token_j in enumerate(tokens):
        distance_rows.append({
            "from_token": token_i,
            "to_token": token_j,
            "rnn_steps": abs(i - j),
            "self_attention_steps": 1 if i != j else 0
        })

distance_df = pd.DataFrame(distance_rows)
distance_df[(distance_df["from_token"] == "Paris") & (distance_df["to_token"] == "Amazon")]

## 6. Create Your Own Attention Pattern

Now create a new attention matrix manually.

Imagine the model strongly connects:

- `Amazon` with `office`
- `Paris` with `office`
- `Paris` with `in`

This kind of pattern could be useful for NER.

In [ ]:
custom_attention = np.ones((len(tokens), len(tokens)))
custom_attention = custom_attention / custom_attention.sum(axis=1, keepdims=True)

# Increase some useful attention weights manually
amazon = tokens.index("Amazon")
office = tokens.index("office")
paris = tokens.index("Paris")
in_token = tokens.index("in")

custom_attention[amazon, office] += 0.4
custom_attention[paris, office] += 0.4
custom_attention[paris, in_token] += 0.2

# Re-normalize each row so rows sum to 1
custom_attention = custom_attention / custom_attention.sum(axis=1, keepdims=True)

plot_attention_matrix(custom_attention, tokens, title="Custom Attention Pattern")

## 7. From Scores to Attention Weights: Softmax

Attention starts with raw similarity scores.

These scores are converted into probabilities using softmax.

Softmax makes all values positive and ensures that they sum to 1.

In [ ]:
def softmax(x):
    x = np.array(x)
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()


scores = np.array([2.0, 1.0, 0.5, 0.1])
weights = softmax(scores)

pd.DataFrame({
    "raw_score": scores,
    "attention_weight": weights
})

## 8. Query, Key, and Value Intuition

In Transformers, each token is projected into three vectors:

- Query: what information am I looking for?
- Key: what information do I contain?
- Value: what information do I provide?

Attention compares queries and keys, then uses the resulting weights to combine values.

In [ ]:
# Tiny numerical example
Q = np.array([[1.0, 0.0]])
K = np.array([
    [1.0, 0.0],
    [0.8, 0.2],
    [0.0, 1.0]
])
V = np.array([
    [10.0, 0.0],
    [8.0, 2.0],
    [0.0, 10.0]
])

scores = Q @ K.T
weights = softmax(scores[0])
output = weights @ V

print("Scores:", scores)
print("Attention weights:", weights)
print("Weighted output:", output)

## 9. Mini Exercise

Use the sentence below:

```text
Google acquired DeepMind in London.
```

Tasks:

1. Create a token list.
2. Create a simple attention matrix.
3. Make `London` attend strongly to `acquired` and `DeepMind`.
4. Visualize the matrix.
5. Explain whether the attention pattern is useful for NER.

In [ ]:
# TODO

exercise_tokens = ["Google", "acquired", "DeepMind", "in", "London", "."]

# Create a uniform attention matrix
exercise_attention = np.ones((len(exercise_tokens), len(exercise_tokens)))
exercise_attention = exercise_attention / exercise_attention.sum(axis=1, keepdims=True)

# Modify the row for London
# Hint: find the index of London, acquired, and DeepMind

# plot_attention_matrix(exercise_attention, exercise_tokens, title="Exercise Attention Matrix")

## 10. Reflection

Answer briefly:

1. What does one row of an attention matrix represent?
2. Why do attention rows usually sum to 1?
3. How is self-attention different from RNN processing?
4. Why is attention useful for NER?
5. What are the limitations of interpreting attention weights?

## Summary

In this notebook, we visualized attention and self-attention.

Main ideas:

- Attention weights show how much one token attends to another.
- Self-attention allows every token to directly access every other token.
- Attention helps models capture long-range dependencies.
- This is one of the core ideas behind Transformers.

Next, we will use pretrained Transformer models for NER.